In [ ]:
import os
import shutil

import cv2
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from torchvision.ops import masks_to_boxes
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    read_image_tensor,
    save_image_tensor,
    show_pytorch_images,
)

data_dir = root / "data"
dataset_name = "FETAL_BRAIN_PLANES"
dataset_dir = data_dir / dataset_name

# Create FETAL_BRAIN_PLANES dataset

## Create Dataset directory

In [ ]:
if not dataset_dir.exists():
    dataset_dir.mkdir(parents=True)
    print(f"Dataset directory '{dataset_name}' created successfully.")
else:
    print(f"Dataset directory '{dataset_name}' already exists.")

## Prepare csv

### Load FETAL_PLANE csv

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes = fetal_planes[
    ["Image_name", "Patient_num", "Brain_plane", "New_brain_plane", "Train", "Subset", "Duplicate", "Valid"]
]
fetal_planes = fetal_planes.reset_index(drop=True)
fetal_planes

In [ ]:
invalid_images = 0
other_images = 0

for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    if not np.isnan(row["Duplicate"]) or row["Valid"] != 1 or isinstance(row["New_brain_plane"], str):
        fetal_planes.loc[index, "Valid"] = 0
        invalid_images += 1

    if row["Brain_plane"] == "Other":
        fetal_planes.loc[index, "Brain_plane"] = "Not A Brain"
        other_images += 1

fetal_planes = fetal_planes.drop(["New_brain_plane", "Duplicate"], axis=1)

print(f"Images: {len(fetal_planes)}")
print(f"Invalid images: {invalid_images}")
print(f"Other images: {other_images}")
fetal_planes

### Validate Subset on fetal_head_segmentation

In [ ]:
fetal_head_segmentation = pd.read_csv(f"{data_dir}/FETAL_HEAD_SEGMENTATION/data.csv")
fetal_head_segmentation

incorrect = 0
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    image_name = row["Image_name"]
    subset = row["Subset"]
    subset_2 = list(fetal_head_segmentation[fetal_head_segmentation["Image_name"] == image_name]["Subset"])
    if len(subset_2) == 1 and subset_2[0] != subset:
        incorrect += 1
        print(f"Error: {index} {row}")
        print(f"subset_2: {subset_2}")
        print()
        break

print(f"incorrect: {incorrect}")

### Statistics

In [ ]:
fetal_planes["Subset"].value_counts()

In [ ]:
fetal_planes["Brain_plane"].value_counts()

### Save csv

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes

##  Prepare Data

### Delete Dataset Image directory

In [ ]:
images_dir = dataset_dir / "Images"

if images_dir.exists():
    shutil.rmtree(images_dir)
    print(f"Directory '{images_dir}' and all its contents have been removed.")
else:
    print(f"Directory '{images_dir}' does not exist.")

### Create Dataset Images directory

In [ ]:
images_dir = dataset_dir / "Images"

if not images_dir.exists():
    shutil.copytree(f"{data_dir}/FETAL_PLANES/Images", images_dir)
    print(f"{images_dir.stem} directory created successfully.")
else:
    print(f"{images_dir.stem} directory already exists.")

### Process images

In [ ]:
def read_mask_tensor(mask_path):
    img_path = data_dir / "FETAL_HEAD_SEGMENTATION" / mask_path
    image = read_image_tensor(img_path)
    image = image[:1, :, :]  # single-channel for mask
    image = image // 255
    return image


def find_angle(mask):
    mask = mask.squeeze(0)

    # Get the indices of the non-zero elements
    coords = torch.nonzero(mask, as_tuple=False).float()
    # Center the data by subtracting the mean
    mean = torch.mean(coords, dim=0)
    centered_coords = coords - mean

    # Calculate the covariance matrix
    covariance_matrix = torch.matmul(centered_coords.T, centered_coords)
    # Perform eigenvalue decomposition to find eigenvectors (principal components)
    eigenvalues, eigenvectors = torch.linalg.eigh(covariance_matrix)
    # The eigenvector corresponding to the largest eigenvalue (the first principal component)
    principal_axis = eigenvectors[:, 1]
    # Calculate the angle of the principal axis
    angle = torch.atan2(principal_axis[0], principal_axis[1]) * 180 / torch.pi

    return angle.round().int()


def crop(image, x1, y1, x2, y2, pad=10):

    pad_x = int((x2 - x1) * (pad / 100.0))
    left_pad = pad_x // 2
    right_pad = pad_x - left_pad

    pad_y = int((y2 - y1) * (pad / 100.0))
    top_pad = pad_y // 2
    bottom_pad = pad_y - top_pad

    x1 = max(x1 - left_pad, 0)
    y1 = max(y1 - top_pad, 0)
    x2 = min(x2 + right_pad, image.shape[2])
    y2 = min(y2 + bottom_pad, image.shape[1])
    return TF.crop(image, y1, x1, y2 - y1, x2 - x1)


def add_half(center, radius):
    if torch.isclose(center, torch.round(center)):
        if torch.isclose(radius, torch.round(radius)):
            radius = radius + 0.5
    else:
        if not torch.isclose(radius, torch.round(radius)):
            radius = radius + 0.5
    return radius


def create_ellipse_tensor(
    height: int,
    width: int,
    center_x: int,
    center_y: int,
    radius_x: int,
    radius_y: int,
) -> torch.Tensor:
    """
    Creates a 2D PyTorch tensor with a binary ellipse shape.

    Args:
        height (int): The height of the tensor.
        width (int): The width of the tensor.
        center_x (int): The x-coordinate of the ellipse's center.
        center_y (int): The y-coordinate of the ellipse's center.
        radius_x (int): The radius of the ellipse along the x-axis.
        radius_y (int): The radius of the ellipse along the y-axis.

    Returns:
        torch.Tensor: A 2D tensor of shape (height, width) with 1s inside the
                      ellipse and 0s outside.
    """
    radius_x = add_half(center_x, radius_x)
    radius_y = add_half(center_y, radius_y)

    y, x = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")
    x_translated = x - center_x
    y_translated = y - center_y
    ellipse_mask = (x_translated / radius_x) ** 2 + (y_translated / radius_y) ** 2 <= 1
    ellipse_tensor = ellipse_mask.float()
    return ellipse_tensor


def get_dice_score(inputs: torch.Tensor, targets: torch.Tensor, smooth: float = 1e-6) -> torch.Tensor:
    inputs = inputs.contiguous().view(-1)
    targets = targets.contiguous().view(-1)

    intersection = (inputs * targets).sum()
    dice_score = (2.0 * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
    return dice_score


def overlap_mask(image: torch.Tensor, mask: torch.Tensor):
    image = image.clone()
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    return image


image_size = (192, 256)
transform = torch.nn.Sequential(
    T.Grayscale(),
    PadToAspectRatio(image_size, fill=0),
    # Resize(image_size, interpolation=T.InterpolationMode.BILINEAR),
    Resize(image_size, interpolation=T.InterpolationMode.NEAREST),
    T.ToDtype(torch.float32, scale=True),
)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

In [ ]:
images_dir = dataset_dir / "Images"

segmentation = pd.read_csv(data_dir / "FETAL_HEAD_SEGMENTATION" / "data.csv", dtype={"Patient_num": str})
fetal_planes = pd.read_csv(f"{dataset_dir}/data.csv")
fetal_planes

In [ ]:
if "Image_crop_name" not in fetal_planes.columns:
    fetal_planes.insert(1, "Image_crop_name", "")
else:
    fetal_planes["Image_crop_name"] = ""


brain_planes = fetal_planes[fetal_planes["Brain_plane"] != "Not A Brain"]
for index, row in tqdm(brain_planes.iterrows(), total=len(brain_planes), desc="Process images"):
    for _, segmentation_row in segmentation[segmentation["Image_name"] == row["Image_name"]].iterrows():
        image_name = row["Image_name"]
        image_path = images_dir / f"{image_name}.png"
        image = read_image_tensor(image_path)
        image = pad_to_aspect_ration(image)

        mask_path = segmentation_row["Segmentation_path"]
        mask = read_mask_tensor(mask_path)

        angle = find_angle(mask)
        mask = pad_to_aspect_ration(mask)
        mask = TF.resize(mask, size=image.shape[1:], interpolation=T.InterpolationMode.NEAREST)
        mask = TF.rotate(mask, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)
        image = TF.rotate(image, angle=angle, expand=True, interpolation=T.InterpolationMode.BILINEAR)

        x1, y1, x2, y2 = masks_to_boxes(mask)[0].int()
        image = crop(image, x1, y1, x2, y2, pad=5)

        output_path = dataset_dir / "Images" / f"{image_name}_crop.png"
        save_image_tensor(image, output_path)
        fetal_planes.loc[index, "Image_crop_name"] = output_path.stem

fetal_planes

### Save csv

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes